# 01 – Data download and baseline benchmarks
Downloads daily prices for AAPL, MSFT, JPM, XOM and SPY (2005–2025) with yfinance, saves them to Drive, and works out the two zero-parameter benchmarks (majority class and persistence).

## 1. Setup

In [ ]:
!pip install yfinance -q

import yfinance as yf
import pandas as pd
import numpy as np
import os
from datetime import datetime
from sklearn.metrics import accuracy_score, matthews_corrcoef, confusion_matrix

# Fixed seed for anything stochastic later in the project.
np.random.seed(42)

print("yfinance", yf.__version__)
print("pandas  ", pd.__version__)
print("Libraries loaded")

In [ ]:
# Mount Google Drive so files survive when the Colab session disconnects.
# A pop-up will ask you to sign in and grant access - click through it.
from google.colab import drive
drive.mount('/content/drive')

BASE    = "/content/drive/MyDrive/CS5500_dissertation"
DATA    = f"{BASE}/data"
RESULTS = f"{BASE}/results"

for folder in (DATA, RESULTS):
    os.makedirs(folder, exist_ok=True)

print("Project folders ready")
print(" ", DATA)
print(" ", RESULTS)

## 2. Configuration
Every decision that could affect a result lives in one place, so the methodology chapter can be written straight from this cell.

In [ ]:
TICKERS     = ["AAPL", "MSFT", "JPM", "XOM", "SPY"]
START_DATE  = "2005-01-01"
END_DATE    = "2025-12-31"
TEST_START  = "2024-01-01"   # held-out period - do not touch before final evaluation

print(f"{len(TICKERS)} assets: {', '.join(TICKERS)}")
print(f"Full period:  {START_DATE} to {END_DATE}")
print(f"Held out:     {TEST_START} onward")

## 3. Download the data
Adjusted close, not raw close.

In [ ]:
def download_ticker(ticker, start, end):
    """Download one ticker and return a clean single-level-column dataframe."""
    df = yf.download(ticker, start=start, end=end,
                     auto_adjust=False, progress=False)

    # Recent yfinance versions return two-level column names even for a
    # single ticker. Flatten before saving so the CSV reads back cleanly.
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = df.columns.get_level_values(0)

    df.index.name = "Date"
    return df


prices = {}

for t in TICKERS:
    df = download_ticker(t, START_DATE, END_DATE)

    if len(df) == 0:
        print(f"{t}: EMPTY - likely rate limited, wait 60s and re-run this cell")
        continue

    df.to_csv(f"{DATA}/{t}.csv")
    prices[t] = df
    print(f"{t}: {len(df):>5} rows  {df.index.min().date()} to {df.index.max().date()}")

print(f"\n{len(prices)}/{len(TICKERS)} downloaded and saved")

## 4. Data quality checks
Row counts, missing values and basic return statistics for each asset.

In [ ]:
checks = []

for t, df in prices.items():
    px  = df["Adj Close"].astype(float)
    ret = px.pct_change().dropna()

    checks.append({
        "ticker":      t,
        "rows":        len(df),
        "missing":     int(df["Adj Close"].isna().sum()),
        "pct_up_days": round((ret > 0).mean(), 4),
        "mean_ret":    round(ret.mean(), 6),
        "daily_vol":   round(ret.std(), 4),
        "worst_day":   round(ret.min(), 4),
        "best_day":    round(ret.max(), 4),
    })

quality = pd.DataFrame(checks)
print(quality.to_string(index=False))

quality.to_csv(f"{RESULTS}/data_quality.csv", index=False)
print(f"\nSaved to {RESULTS}/data_quality.csv")

## 5. Benchmark 1 — majority class
The simplest possible rule: always predict *up*, because markets drift upward over time.

In [ ]:
def build_direction_frame(df, test_start):
    """Return today's return and tomorrow's direction, restricted to the test period."""
    px  = df["Adj Close"].astype(float)
    ret = px.pct_change()

    frame = pd.DataFrame({
        "ret_today":  ret,
        "up_today":   (ret > 0).astype(int),
        "up_tomorrow": (ret.shift(-1) > 0).astype(int),
    }).dropna()

    return frame.loc[test_start:]


def score(y_true, y_pred, label, ticker):
    """Accuracy with a 95% CI, plus MCC."""
    acc = accuracy_score(y_true, y_pred)
    mcc = matthews_corrcoef(y_true, y_pred)
    se  = np.sqrt(acc * (1 - acc) / len(y_true))

    return {
        "ticker":   ticker,
        "model":    label,
        "n":        len(y_true),
        "accuracy": round(acc, 4),
        "ci_low":   round(acc - 1.96 * se, 4),
        "ci_high":  round(acc + 1.96 * se, 4),
        "mcc":      round(mcc, 4),
    }


results = []

for t, df in prices.items():
    test = build_direction_frame(df, TEST_START)

    # Always predict "up"
    always_up = np.ones(len(test), dtype=int)
    results.append(score(test["up_tomorrow"], always_up, "majority_class", t))

majority = pd.DataFrame(results)
print(majority.to_string(index=False))

## 6. Benchmark 2 — naive persistence
"Tomorrow moves in the same direction as today." This uses no features and no learning, yet it is the benchmark most stock-prediction papers fail to include.

In [ ]:
for t, df in prices.items():
    test = build_direction_frame(df, TEST_START)
    results.append(
        score(test["up_tomorrow"], test["up_today"], "persistence", t)
    )

benchmarks = pd.DataFrame(results)

pivot = benchmarks.pivot(index="ticker", columns="model",
                         values=["accuracy", "mcc"])

print(benchmarks.to_string(index=False))
print("\nSide by side:\n")
print(pivot.to_string())

benchmarks.to_csv(f"{RESULTS}/baseline_benchmarks.csv", index=False)
print(f"\nSaved to {RESULTS}/baseline_benchmarks.csv")

## 7. Confusion matrices
A worked example for one asset, showing *how* persistence gets the accuracy it does.

In [ ]:
EXAMPLE = "AAPL"

test = build_direction_frame(prices[EXAMPLE], TEST_START)
cm = confusion_matrix(test["up_tomorrow"], test["up_today"])

print(f"{EXAMPLE} - naive persistence, test period from {TEST_START}\n")
print(pd.DataFrame(
    cm,
    index=["actual down", "actual up"],
    columns=["predicted down", "predicted up"]
).to_string())

print(f"\nActual up days:    {test['up_tomorrow'].mean():.1%}")
print(f"Predicted up days: {test['up_today'].mean():.1%}")

## 8. Write the results log
Every experiment gets a dated entry.

In [ ]:
today = datetime.now().strftime("%Y-%m-%d")

entry = f"""# CS5500 Results Log

## {today} - Notebook 01: data acquisition and baseline benchmarks

**Data**
- Source: yfinance, daily OHLCV
- Period: {START_DATE} to {END_DATE}
- Price series: adjusted close (accounts for splits and dividends)
- Assets: {', '.join(TICKERS)}
- Held-out test period: {TEST_START} onward

**Data quality**

{quality.to_string(index=False)}

**Baseline benchmarks (test period only)**

{benchmarks.to_string(index=False)}

**Interpretation**
- MCC is approximately zero for both benchmarks across all assets, indicating
  no meaningful association between prediction and outcome.
- Majority class achieves non-trivial accuracy purely from upward drift in the
  test period. Any model reporting accuracy at or below this level has
  demonstrated nothing.
- Persistence accuracy sits slightly above 0.50 but the 95% confidence
  intervals span 0.50, so this is not evidence of a genuine edge.
- These two figures are the bar for every subsequent model.

**Next**
- Notebook 02: feature engineering with strictly backward-looking windows
- Notebook 03: expanding-window walk-forward splitter
"""

with open(f"{BASE}/results_log.md", "w") as f:
    f.write(entry)

print(entry)